In [69]:
# from jupytergis import GISDocument
from jupytergis.tiler import GISDocument
from typing import Iterable, List
from pyproj import Transformer
import pystac_client
import stackstac

doc = GISDocument("local.jGIS")
doc

In [70]:
def extent_3857_to_bbox_4326(extent: Iterable[float]) -> List[float]:
    """
    Convert a Web Mercator extent to a STAC-ready lon/lat bbox.
    Input extent format: [minx, miny, maxx, maxy] in EPSG:3857
    Output bbox format:  [min_lon, min_lat, max_lon, max_lat] in EPSG:4326
    """
    minx, miny, maxx, maxy = extent
    transformer = Transformer.from_crs(
        "EPSG:3857",
        "EPSG:4326",
        always_xy=True,  # keep x=lon, y=lat ordering
    )
    min_lon, min_lat = transformer.transform(minx, miny)
    max_lon, max_lat = transformer.transform(maxx, maxy)
    return [min_lon, min_lat, max_lon, max_lat]

In [86]:
m = doc.extent
# Full map extent in lon/lat (for reference only).
bbox_doc = (
    extent_3857_to_bbox_4326([m.west, m.south, m.east, m.north]) if m else None
)
# STAC search and stackstac.stack must use the *same* bounds; otherwise the
# stack is huge and mostly empty while items only cover a small area.
bbox_search = [-2.98, 47.50, -2.58, 47.66]
bbox = bbox_search
bbox_doc, bbox

([-3.1010902936421885,
  47.08831574172623,
  -2.215258999938924,
  47.83623140747555],
 [-2.98, 47.5, -2.58, 47.66])

In [72]:
URL = "https://earth-search.aws.element84.com/v1"
catalog = pystac_client.Client.open(URL)

In [73]:
%%time
items = catalog.search(
    bbox=bbox,
    collections=["sentinel-2-l2a"],
    datetime="2020-03-01/2020-03-02",
).item_collection()
len(items)

CPU times: user 5.66 ms, sys: 1.06 ms, total: 6.72 ms
Wall time: 986 ms


4

In [74]:
%time 
da = stackstac.stack(items, epsg=4326, bounds_latlon=bbox)

CPU times: user 2 μs, sys: 0 ns, total: 2 μs
Wall time: 3.58 μs


In [75]:
state = {"clipped": None}
sub_id = doc.bind_extent_slice_callback(
    da,
    on_slice=lambda c: state.update({"clipped": c}),
    data_crs="EPSG:4326",
    x_name="x",
    y_name="y",
)


In [88]:
green = state["clipped"].sel(band='green')
nir = state["clipped"].sel(band='nir')
ndwi = (green - nir) / (green + nir)
ndwi = ndwi.where((green + nir) != 0)
ndwi.name = 'ndwi'
ndwi

<xarray.DataArray 'ndwi' (time: 4, y: 1766, x: 2994)> Size: 169MB
dask.array<where, shape=(4, 1766, 2994), dtype=float64, chunksize=(1, 1024, 1024), chunktype=numpy.ndarray>
Coordinates: (12/47)
  * time                                     (time) datetime64[ns] 32B 2020-0...
    id                                       (time) <U24 384B 'S2B_30TWT_2020...
    s2:medium_proba_clouds_percentage        (time) float64 32B 6.791 ... 19.86
    earthsearch:s3_path                      (time) <U79 1kB 's3://sentinel-c...
    grid:code                                (time) <U10 160B 'MGRS-30TWT' .....
    s2:unclassified_percentage               (time) float64 32B 6.041 ... 0.2821
    ...                                       ...
    mgrs:latitude_band                       <U1 4B 'T'
    proj:code                                <U10 40B 'EPSG:32630'
    constellation                            <U10 40B 'sentinel-2'
    raster:bands                             object 8B None
    gsd                                      object 8B 10
    epsg                                     int64 8B 4326
Attributes:
    spec:           RasterSpec(epsg=4326, bounds=(-2.9800768041580588, 47.499...
    crs:            epsg:4326
    transform:      | 0.00, 0.00,-2.98|\n| 0.00,-0.00, 47.66|\n| 0.00, 0.00, ...
    resolution_xy:  (0.00013365371144808983, 9.065880006519783e-05)

In [90]:
import dask.diagnostics
with dask.diagnostics.ProgressBar():
    data = ndwi.compute()

[########################################] | 100% Completed | 4.43 sms


In [91]:
# Rescale for TiTiler: use the same 2D NDWI you display, not raw multi-band data.
ndwi2 = data.isel(time=0)
q = ndwi2.quantile([0.02, 0.98], skipna=True).compute()
vmin = float(q.sel(quantile=0.02).item())
vmax = float(q.sel(quantile=0.98).item())
# Fallback if the scene is all-NaN (clouds / no overlap).
# import math

# if not (math.isfinite(vmin) and math.isfinite(vmax)) or vmin == vmax:
#     vmin, vmax = -1.0, 1.0
# vmin_acc, vmax_acc = vmin, vmax
vmin, vmax

(-0.8005249343832022, 0.739010989010989)

In [92]:
# Scalar stats on 2D NDWI (avoid int() on NaN).
_mn = float(ndwi2.min(skipna=True).compute().item())
_mx = float(ndwi2.max(skipna=True).compute().item())
_mn, _mx

(-0.9263331500824628, 0.99818676337262)

In [93]:
await doc.add_tiler_layer(
    name="test",
    data_array=data.isel(time=0),
    colormap_name="plasma",
    rescale=(vmin, vmax),
    reproject="max",
)

'e0e7a7aa-a556-49e2-994b-7a2ef5ee5c52'